# 04 - CAD e Prototipagem
> Scripts OpenSCAD, otimizacao para impressao 3D e parametros de fabricacao

**Modulo:** `11_engfis_tecnico` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=2048):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

# System prompt especializado para CAD e prototipagem
CAD_SYSTEM = (
    'Voce e um engenheiro mecanico especialista em CAD, impressao 3D '
    'e prototipagem rapida. Responda em portugues, sem acentos. '
    'Seja preciso com dimensoes, tolerancias e parametros de fabricacao.'
)

def ask_cad(prompt, model=SONNET, max_tokens=2048):
    return ask(prompt, system=CAD_SYSTEM, model=model, max_tokens=max_tokens)

print('Pronto!')

## Gerando Scripts OpenSCAD com Claude

OpenSCAD e uma ferramenta de modelagem 3D programatica -- ideal para
prototipagem parametrica. Em vez de clicar em interfaces graficas,
escrevemos codigo que define a geometria. Claude pode gerar esses
scripts a partir de descricoes textuais.

In [ ]:
# Pedir ao Claude para gerar um enclosure parametrico em OpenSCAD
prompt_enclosure = """
Gere um script OpenSCAD completo para uma caixa (enclosure) parametrica com:
- Dimensoes configuraveis: largura, comprimento, altura, espessura da parede
- Tampa removivel com encaixe macho-femea
- 4 furos para parafusos M3 nos cantos (com bosses/pilares internos)
- Cantos arredondados (fillet)
- Ventilacao (slots laterais)

Use boas praticas: variaveis no topo, modulos separados para cada componente,
comentarios explicando cada secao. Os parametros default devem ser para uma
caixa de 80x60x35mm com paredes de 2mm.
"""

scad_code = ask_cad(prompt_enclosure, max_tokens=3000)
print(scad_code)

In [ ]:
# Salvar o script gerado para uso no OpenSCAD
with open('enclosure_parametrico.scad', 'w') as f:
    f.write(scad_code)

print('Script salvo em enclosure_parametrico.scad')
print('Abra no OpenSCAD para visualizar e renderizar.')
print()

# Pedir variacao -- mudar parametros
variacao = ask_cad(
    'Usando o script OpenSCAD anterior como base, quais linhas eu preciso '
    'alterar para fazer uma versao de 120x80x50mm com parede de 3mm e '
    'furos para parafusos M4 em vez de M3? Liste apenas as mudancas.'
)
print('=== Modificacoes necessarias ===')
print(variacao)

**Abordagem parametrica**: Todas as dimensoes sao variaveis no topo do
script. Para criar uma nova versao, basta mudar os valores -- sem
redesenhar nada. Isso e fundamental para prototipagem iterativa.

## Design para Impressao 3D

Nem todo modelo CAD e imprimivel. Impressoras FDM (filamento) tem
limitacoes fisicas que afetam o design. Claude pode revisar um
projeto e apontar problemas antes de gastar filamento e tempo.

In [ ]:
# Consultar regras de design para FDM
regras = ask_cad("""
Liste as regras fundamentais de Design for Manufacturing (DFM) para
impressao 3D FDM. Para cada regra, inclua:
- O limite numerico (ex: angulo maximo, espessura minima)
- Por que existe essa limitacao fisica
- Como contornar quando necessario

Cubra: espessura de parede, overhangs, bridging, suportes, tolerancias
de encaixe, orientacao de impressao, e warping.
""")
print(regras)

In [ ]:
# Pedir revisao de um design especifico
descricao_peca = """
Preciso imprimir um suporte de sensor para um projeto de laboratorio:
- Braco em L com 120mm de comprimento horizontal e 80mm vertical
- Secao transversal retangular de 8mm x 4mm
- Furo passante de 5mm na ponta horizontal para fixar sensor (peso: 50g)
- Base com 2 furos M4 para fixacao em perfil de aluminio
- Angulo de 90 graus entre braco horizontal e vertical
- Material: PLA
- Impressora: Ender 3 (bico 0.4mm)
"""

revisao = ask_cad(f"""
Revise este design para impressao 3D FDM e identifique problemas:

{descricao_peca}

Para cada problema encontrado:
1. Descreva o problema
2. Explique o impacto (falha, qualidade ruim, etc)
3. Sugira uma solucao concreta com dimensoes

Considere: orientacao de impressao, necessidade de suporte,
resistencia mecanica, e tolerancias.
""")
print(revisao)

## Otimizando Parametros de Fatiamento

O slicer (fatiador) converte o modelo 3D em instrucoes para a
impressora. Os parametros de fatiamento afetam drasticamente
a qualidade, resistencia e tempo de impressao.

In [ ]:
# Tabela comparativa de parametros por material e uso
parametros = ask_cad("""
Crie uma tabela comparativa de parametros de fatiamento para impressao 3D FDM.

Materiais: PLA, PETG, ABS
Casos de uso: peca estrutural, peca estetica, prototipo rapido

Para cada combinacao material + caso de uso, recomende:
- Altura de camada (mm)
- Numero de perimetros (walls)
- Infill (%) e padrao (grid, gyroid, etc)
- Velocidade de impressao (mm/s)
- Temperatura do bico (C)
- Temperatura da mesa (C)
- Cooling fan (%)
- Retraction (mm e mm/s)

Formato: tabela organizada, facil de consultar.
Bico: 0.4mm padrao.
""")
print(parametros)

In [ ]:
# Consulta especifica para uma peca
config_especifica = ask_cad("""
Preciso imprimir uma engrenagem de teste em PETG:
- Diametro externo: 40mm
- 20 dentes, modulo 1.5
- Espessura: 8mm
- Precisa de boa precisao dimensional (tolerancia < 0.2mm)
- Vai sofrer carga mecanica moderada
- Impressora: Ender 3 com direct drive

Recomende os parametros de fatiamento otimizados, explicando
por que cada escolha e importante para esta peca especifica.
Inclua tambem a orientacao de impressao ideal.
""")
print(config_especifica)

## G-code: Entendendo e Modificando

G-code e a linguagem que a impressora 3D realmente executa.
Entender G-code permite fazer ajustes finos que o slicer nao
oferece, como pausas para inserir componentes ou mudancas
de temperatura em camadas especificas.

In [ ]:
# Trecho de G-code para Claude explicar
gcode_trecho = """
G28 ; Home all axes
G29 ; Auto bed leveling
M104 S210 ; Set hotend temp
M140 S60 ; Set bed temp
M190 S60 ; Wait for bed temp
M109 S210 ; Wait for hotend temp
G92 E0 ; Reset extruder
G1 Z2.0 F3000 ; Move Z up
G1 X0.1 Y20 Z0.3 F5000.0 ; Move to start
G1 X0.1 Y200.0 Z0.3 F1500.0 E15 ; Draw purge line
G1 X0.4 Y200.0 Z0.3 F5000.0 ; Move to side
G1 X0.4 Y20 Z0.3 F1500.0 E30 ; Draw 2nd purge line
G92 E0 ; Reset extruder
G1 Z2.0 F3000 ; Move Z up
;LAYER:0
G0 F6000 X10.5 Y10.5 Z0.2
G1 F1200 X50.5 Y10.5 E2.134
G1 X50.5 Y50.5 E4.268
"""

explicacao = ask_cad(f"""
Explique cada linha deste G-code de impressao 3D, comando por comando.
Para cada linha, diga:
- O que o comando faz
- O significado de cada parametro (G, M, X, Y, Z, F, E, S)
- Por que esta nesta sequencia

G-code:
{gcode_trecho}
""")
print(explicacao)

In [ ]:
# Pedir modificacoes no G-code
modificacao = ask_cad("""
Gere trechos de G-code para as seguintes modificacoes em uma impressao:

1. PAUSA NA CAMADA 25 para inserir um ima (6x3mm) dentro da peca:
   - Mover o bico para fora da peca
   - Pausar e esperar confirmacao do usuario
   - Retomar impressao

2. MUDANCA DE TEMPERATURA na camada 50:
   - Reduzir temperatura do bico de 210C para 195C (parte estetica)
   - Reduzir velocidade em 20%

3. TROCA DE FILAMENTO na camada 75:
   - Retrair filamento
   - Mover para posicao de troca
   - Purgar novo filamento
   - Retomar

Para cada trecho, inclua comentarios explicativos e indique
onde inserir no G-code original (antes/depois de ;LAYER:XX).
""")
print(modificacao)

## Calculo Estrutural para Prototipos

Pecas impressas em 3D tem propriedades mecanicas anisotropicas --
a resistencia depende da direcao das camadas. Claude pode ajudar
com calculos basicos de tensao e verificacao de carga.

In [ ]:
# Analise estrutural basica de uma peca impressa
analise = ask_cad("""
Faca uma analise estrutural simplificada para esta peca impressa em 3D:

PECA: Braco de suporte em L (viga em balanco)
- Comprimento horizontal: 100mm
- Secao transversal: 15mm x 10mm (largura x altura)
- Material: PETG
- Infill: 50% gyroid
- 3 perimetros (0.4mm cada)
- Carga na ponta: 2 kg (forca = ~19.6 N)
- Orientacao de impressao: braco horizontal impresso na horizontal
  (camadas perpendiculares ao comprimento)

Calcule:
1. Tensao maxima de flexao (considere secao efetiva com infill)
2. Fator de seguranca (usando propriedades tipicas de PETG impresso)
3. Deflexao estimada na ponta
4. Modo de falha mais provavel
5. Recomendacoes para aumentar resistencia se necessario

Mostre as formulas e calculos passo a passo.
Use propriedades tipicas de PETG impresso (nao bulk).
""")
print(analise)

In [ ]:
# Comparacao de materiais para aplicacao especifica
comparacao = ask_cad("""
Compare as propriedades mecanicas de pecas impressas em 3D (FDM) para:
PLA, PETG, ABS, ASA, Nylon (PA6), e PC (policarbonato).

Para cada material, liste:
- Resistencia a tracao (MPa) -- valores TIPICOS para pecas impressas
- Modulo de elasticidade (GPa)
- Resistencia ao impacto (qualitativa: baixa/media/alta)
- Temperatura maxima de uso continuo (C)
- Resistencia a UV
- Facilidade de impressao (1-5)
- Melhor aplicacao tipica

IMPORTANTE: Use valores para pecas IMPRESSAS, nao valores de datasheet
do material bulk. Pecas impressas sao significativamente mais fracas
entre camadas.
""")
print(comparacao)

## Documentacao Tecnica Automatizada

Todo prototipo precisa de documentacao: especificacoes, lista de
materiais, instrucoes de montagem. Claude pode gerar documentacao
tecnica estruturada a partir de descricoes informais.

In [ ]:
# Gerar documentacao tecnica completa
descricao_projeto = """
Estou fazendo um case para Raspberry Pi 4 com:
- Corpo principal impresso em PETG preto
- Tampa superior com ventilacao
- Suporte interno para camera Pi Camera v2
- Slot para cartao SD acessivel
- Furos para todos os conectores (USB, HDMI, ethernet, GPIO)
- Montagem em parede com furos para parafusos
- Ventilador 30x30mm com grade
- Espaco para heatsinks no SoC e RAM
"""

doc_tecnica = ask_cad(f"""
Gere uma especificacao tecnica completa para este prototipo:

{descricao_projeto}

A documentacao deve incluir:

1. RESUMO DO PROJETO
   - Objetivo, aplicacao, revisao

2. DIMENSOES GERAIS
   - Todas as dimensoes externas e internas relevantes
   - Tolerancias para cada tipo de feature

3. BILL OF MATERIALS (BOM)
   - Todas as pecas (impressas e compradas)
   - Quantidade, material, fornecedor sugerido

4. ESPECIFICACOES DE FABRICACAO
   - Parametros de impressao para cada peca
   - Pos-processamento necessario

5. INSTRUCOES DE MONTAGEM
   - Sequencia passo a passo
   - Ferramentas necessarias

6. VERIFICACAO E TESTES
   - Checklist de qualidade
   - Testes funcionais

Formato profissional, pronto para revisao de engenharia.
""", max_tokens=4000)
print(doc_tecnica)

In [ ]:
# Salvar documentacao
with open('spec_rpi4_case.md', 'w') as f:
    f.write(doc_tecnica)

print('Documentacao salva em spec_rpi4_case.md')
print(f'Tamanho: {len(doc_tecnica)} caracteres')

## Exercicios

### Exercicio 1 -- Suporte Parametrico
Peca ao Claude para gerar um script OpenSCAD de um suporte de celular
ajustavel (angulo variavel, largura configuravel para diferentes modelos).

### Exercicio 2 -- Revisao de Design
Descreva uma peca que voce precisa imprimir (real ou imaginaria) e peca
ao Claude para revisar o design. Implemente as sugestoes.

### Exercicio 3 -- Analise de G-code
Exporte o G-code de uma peca no Cura ou PrusaSlicer. Extraia as
primeiras 50 linhas e peca ao Claude para explicar. Depois, adicione
uma pausa na camada 10.

### Exercicio 4 -- Calculo de Carga
Projete um gancho de parede impresso em PLA que suporte 5 kg.
Use Claude para calcular as dimensoes minimas necessarias e
gere o script OpenSCAD correspondente.

### Exercicio 5 -- Pipeline Completo
Escolha um projeto completo (ex: case para Arduino, suporte de
sensor, adaptador mecanico). Use Claude para: (a) gerar o design
OpenSCAD, (b) revisar para impressao, (c) recomendar parametros
de fatiamento, (d) gerar a documentacao tecnica.

In [ ]:
# Seu codigo aqui


## Proximos passos

- Instale o [OpenSCAD](https://openscad.org/) para testar os scripts gerados
- Experimente o [Cura](https://ultimaker.com/software/ultimaker-cura) ou [PrusaSlicer](https://www.prusa3d.com/prusaslicer/) para fatiamento
- Explore a integracao com FreeCAD via Python scripting
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)